In [22]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

In [ ]:
# Dataset paths
cat_path = "cat"
dog_path = "dog"

print("Cat dataset:", len(os.listdir(cat_path)))
print("Dog dataset:", len(os.listdir(dog_path)))

In [ ]:
# Costant values
latent_dim = 128
batch_size = 64
lr = 0.0002
epochs = 400
channels = 3
img_size = 64
betas = (0.5, 0.999)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# Loading and Transforming images
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.CenterCrop(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * channels, [0.5] * channels)
])

dataset_G = datasets.ImageFolder("cat", transform=transform)
dataset_D = datasets.ImageFolder("dog", transform=transform)

dataloader_G = DataLoader(dataset_G, batch_size=batch_size, shuffle=True)
dataloader_D = DataLoader(dataset_D, batch_size=batch_size, shuffle=True)

In [ ]:
#Generator
def gen(latent_dim, channels):
    model = nn.Sequential(
        nn.ConvTranspose2d(latent_dim, 512, 4, 1, 0),
        nn.BatchNorm2d(512),
        nn.ReLU(),
        nn.ConvTranspose2d(512, 256, 4, 2, 1),
        nn.BatchNorm2d(256),
        nn.ReLU(),
        nn.ConvTranspose2d(256, 128, 4, 2, 1),
        nn.BatchNorm2d(128),
        nn.ReLU(),
        nn.ConvTranspose2d(128, 64, 4, 2, 1),
        nn.BatchNorm2d(64),
        nn.ReLU(),

        nn.ConvTranspose2d(64 , channels, 4, 2, 1),
        nn.Tanh()

    )
    return model

In [ ]:
#Discriminator
def disc(channels):
    model = nn.Sequential(
        nn.Conv2d(channels, 64, 4, 2, 1),
        nn.LeakyReLU(0.2),
        nn.Conv2d(64, 128, 4, 2, 1),
        nn.BatchNorm2d(128),
        nn.LeakyReLU(0.2),
        nn.Conv2d(128, 256, 4, 2, 1),
        nn.BatchNorm2d(256),
        nn.LeakyReLU(0.2),
        nn.Conv2d(256, 512, 4, 2, 1),
        nn.BatchNorm2d(512),
        nn.LeakyReLU(0.2),

        nn.Conv2d(512, 1, 4, 1, 0),
        nn.Sigmoid()
    )

    return model

In [28]:
G = gen(latent_dim, channels).to(device)
D = disc(channels).to(device)

criterion = nn.BCELoss()

op_g = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
op_d = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))

In [ ]:
# Training Loop
losses_D = []
losses_G = []
for e in range(epochs):
    for (d_imgs, _) , (c_imgs, _) in zip(dataloader_D, dataloader_G):
        d_imgs = d_imgs.to(device)
        c_imgs = c_imgs.to(device)
        batch_size = d_imgs.size(0)

        d_labels = torch.ones(batch_size, device=device)
        c_labels = torch.zeros(batch_size, device=device)

        # Train discriminator
        op_d.zero_grad()
        d_output = D(d_imgs).view(-1)
        loss_d = criterion(d_output, d_labels)

        z = torch.randn(batch_size, latent_dim, 1, 1, device=device)

        gen_imgs = G(z)
        c_output = D(gen_imgs.detach()).view(-1)
        loss_c = criterion(c_output, c_labels)

        loss_D = loss_d + loss_c
        loss_D.backward()
        op_d.step()

        # Train generator
        op_g.zero_grad()

        c_output = D(gen_imgs).view(-1)
        loss_G = criterion(c_output, d_labels)

        loss_G.backward()
        op_g.step()
        losses_D.append(loss_D.item())
        losses_G.append(loss_G.item())

    print(f"epoch {e+1}/{epochs} | D: {loss_D.item():.4f} | G: {loss_G.item():.4f}")

    torch.save(G.state_dict(), "cat/generator.pth")
    torch.save(D.state_dict(), "dog/discriminator.pth")



In [ ]:
torch.save(G.state_dict(), "generator.pth")
torch.save(D.state_dict(), "discriminator.pth")

In [ ]:
# Loss Curve
import matplotlib.pyplot as plt

epochs = range(1, len(losses_D) + 1)

plt.figure(figsize=(12, 5))
plt.plot(epochs, losses_D, label="Discriminator Loss")
plt.plot(epochs, losses_G, label="Generator Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("DCGAN Training Losses (400 Epochs)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#Generating Results
G.eval()

with torch.no_grad():
    z = torch.randn(16, latent_dim, 1, 1, device=device)
    gen_imgs = G(z).cpu()

gen_imgs  = (gen_imgs + 1)/2

f,ax = plt.subplots(4,4, figsize=(6,6))

for i, a in enumerate(ax.flat):
    a.imshow(gen_imgs[i].permute(1, 2, 0))
    a.axis("off")

plt.show()